# limiteddepkit quickstart

This notebook is a package-scoped tour of `limiteddepkit`: binary response, ordinal response, count models, and post-estimation checks. It uses deterministic simulated data so the notebook is safe to run on Kaggle or Google Colab without external datasets or credentials.

What this notebook is: a reproducible adoption/demo artifact.  
What it is not: a Stata/R parity certificate or a replacement for the package validation harness.

## Install

Kaggle and Colab runtimes are usually clean. If you are running from a local checkout, skip this cell and make sure the source tree is on `PYTHONPATH`.

In [ ]:
%pip install -q limiteddepkit

## Shared imports and deterministic data helpers

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit

from limiteddepkit import (
    BinaryLogit,
    BinaryProbit,
    OrderedLogit,
    OrderedProbit,
    PoissonRegressor,
    NegativeBinomial,
)

SEED = 20260730
rng = np.random.default_rng(SEED)
print("limiteddepkit import OK")

## 1. Binary response: logit and probit

A binary-response model is the canonical limited-dependent-variable starting point. Here the latent index has two covariates and a constant. We fit logit and probit to the same design matrix, then compare coefficient direction, predicted probabilities, and average marginal effects.

In [ ]:
n = 700
X_binary = pd.DataFrame({
    "const": 1.0,
    "x1": rng.normal(size=n),
    "x2": rng.normal(size=n),
})
beta_binary = np.array([-0.35, 0.85, -0.45])
y_binary = rng.binomial(1, expit(X_binary.to_numpy() @ beta_binary))

logit = BinaryLogit().fit(X_binary, y_binary)
probit = BinaryProbit().fit(X_binary, y_binary)

binary_summary = pd.concat(
    {
        "logit": logit.summary_frame()[["coef", "std_err"]],
        "probit": probit.summary_frame()[["coef", "std_err"]],
    },
    axis=1,
)

print("logit converged:", logit.converged)
print("probit converged:", probit.converged)
display(binary_summary.round(4))

display(logit.predict_proba(X_binary.iloc[:8]).round(4))
display(logit.average_marginal_effects(X_binary).round(4).to_frame("logit_AME"))

assert logit.converged and probit.converged
assert logit.params["x1"] > 0 and logit.params["x2"] < 0
assert np.allclose(logit.predict_proba(X_binary.iloc[:20]).sum(axis=1), 1.0)

## 2. Ordinal response: ordered logit and ordered probit

Ordered models are useful when the outcome has a meaningful rank, such as low/medium/high adoption or ordered survey responses. The notebook checks that thresholds are ordered and predicted category probabilities sum to one.

In [ ]:
n_ord = 900
X_ord = pd.DataFrame({
    "x1": rng.normal(size=n_ord),
    "x2": rng.normal(size=n_ord),
})
beta_ord = np.array([0.9, -0.6])
thresholds = np.array([-0.8, 0.7])
latent = X_ord.to_numpy() @ beta_ord
cum = expit(thresholds[None, :] - latent[:, None])
prob_ord = np.column_stack([cum[:, 0], cum[:, 1] - cum[:, 0], 1.0 - cum[:, 1]])
y_ord = np.array([rng.choice(3, p=row) for row in prob_ord])

ologit = OrderedLogit().fit(X_ord, y_ord)
oprobit = OrderedProbit().fit(X_ord, y_ord)

ordinal_compare = pd.DataFrame({
    "ordered_logit": ologit.params,
    "ordered_probit": oprobit.params,
})

print("ordered logit converged:", ologit.converged)
print("ordered probit converged:", oprobit.converged)
display(ordinal_compare.round(4))
display(ologit.predict_proba(X_ord.iloc[:8]).round(4))
display(ologit.average_marginal_effects(X_ord).round(4))

assert ologit.converged and oprobit.converged
assert np.all(np.diff(ologit.thresholds.to_numpy()) > 0)
assert np.allclose(ologit.predict_proba(X_ord.iloc[:25]).sum(axis=1), 1.0)

## 3. Count outcomes: Poisson and negative binomial

Count models are useful for non-negative integer outcomes such as events, visits, citations, claims, or failures. The negative binomial model adds dispersion beyond Poisson.

In [ ]:
n_count = 650
X_count = pd.DataFrame({
    "const": 1.0,
    "x": rng.normal(size=n_count),
})
offset = pd.Series(rng.normal(scale=0.10, size=n_count))
exposure = pd.Series(rng.uniform(0.5, 2.0, size=n_count))
mean = np.exp(X_count.to_numpy() @ np.array([-0.15, 0.45]) + offset) * exposure

y_pois = pd.Series(rng.poisson(mean))
alpha = 0.70
y_nb = pd.Series(rng.negative_binomial(1.0 / alpha, 1.0 / (1.0 + alpha * mean)))

pois = PoissonRegressor().fit(X_count, y_pois, offset=offset, exposure=exposure)
nb = NegativeBinomial().fit(X_count, y_nb, offset=offset, exposure=exposure)

count_summary = pd.concat(
    {
        "poisson": pois.summary_frame()[["coef", "std_err"]],
        "negative_binomial": nb.summary_frame()[["coef", "std_err"]],
    },
    axis=1,
)

display(count_summary.round(4))
display(pd.DataFrame({
    "poisson_mean_prediction": pois.predict(X_count.iloc[:8], offset=offset.iloc[:8], exposure=exposure.iloc[:8]),
    "nb_mean_prediction": nb.predict(X_count.iloc[:8], offset=offset.iloc[:8], exposure=exposure.iloc[:8]),
}).round(4))

assert pois.converged and nb.converged
assert np.all(pois.predict(X_count.iloc[:20], offset=offset.iloc[:20], exposure=exposure.iloc[:20]) > 0)
assert np.all(nb.predict(X_count.iloc[:20], offset=offset.iloc[:20], exposure=exposure.iloc[:20]) > 0)

## 4. Interpretation checklist

For a public notebook, keep the claims modest and reproducible:

- binary and ordinal predicted probabilities obey probability identities;
- count predictions are positive and respect offset/exposure inputs;
- convergence and inference flags are shown rather than hidden;
- parity with Stata/R belongs in the package validation harness, not in this lightweight cloud notebook.